# Multi-station inventory — `20180530_0225.uom.all.sud`

Reads a PC-SUDS file containing multiple stations, builds an ObsPy `Inventory`,
and prints the scalar sensitivity for every channel.

In [5]:
import sudspy

PATH = "../data/multi_stations/20180530_0225.uom.all.sud"

## Build the inventory

In [6]:
inv = sudspy.read_suds_inv(PATH)
print(inv)

Inventory created at 2026-04-02T02:16:15.581578Z
	Created by: ObsPy 1.4.2
		    https://www.obspy.org
	Sending institution: SUDS
	Contains:
		Networks (22):
			AB (13x), AU (2x), UM (4x), XX (3x)
		Stations (22):
			AB.BEST (BEST)
			AB.BRIG (BRIG)
			AB.CLIF (CLIF)
			AB.FORG (FORG)
			AB.HDDL (HDDL)
			AB.HOLS (HOLS)
			AB.LOCU (LOCU)
			AB.MARD (MARD)
			AB.NARR (NARR)
			AB.OUTU (OUTU)
			AB.SOMU (SOMU)
			AB.WPNH (WPNH)
			AB.WPSH (WPSH)
			AU.GVL (GVL)
			AU.TOO (TOO)
			UM.DDNE (DDNE)
			UM.DDSW (DDSW)
			UM.DDWK (DDWK)
			UM.STBK (STBK)
			XX.LOYU (LOYU)
			XX.SGWU (SGWU)
			XX.TRPU (TRPU)
		Channels (62):
			AB.BEST..c01, AB.BEST..c02, AB.BEST..c03, AB.BRIG..c01, 
			AB.BRIG..c02, AB.BRIG..c03, AB.CLIF..c01, AB.CLIF..c02, 
			AB.CLIF..c03, AB.FORG..c01, AB.FORG..c02, AB.FORG..c03, 
			AB.HDDL..c01, AB.HDDL..c02, AB.HDDL..c03, AB.HOLS..c01, 
			AB.HOLS..c02, AB.HOLS..c03, AB.LOCU..c03, AB.MARD..c01, 
			AB.MARD..c02, AB.MARD..c03, AB.NARR..c01, AB.NARR..c02, 
			AB.NARR..c03, A

## Equipment — sensor & data logger\n\n`read_suds_inv` now parses the `SensorA`, `SensorASerial`, and `DataLogger` fields\nfrom the COMMENT blocks and attaches them to each channel as ObsPy `Equipment` objects\n(`chan.sensor` / `chan.data_logger`)."

In [7]:
print(f"{'Channel':<26}  {'Sensor':<36}  {'Serial':<12}  {'Data logger'}")
print("-" * 90)

for net in inv.networks:
    for sta in net.stations:
        for cha in sta.channels:
            seed_id = f"{net.code}.{sta.code}.{cha.location_code}.{cha.code}"
            sensor  = cha.sensor.description       if cha.sensor      else "-"
            sn      = cha.sensor.serial_number     if cha.sensor      else "-"
            logger  = cha.data_logger.description  if cha.data_logger else "-"
            print(f"{seed_id:<26}  {sensor:<36}  {str(sn):<12}  {logger}")

Channel                     Sensor                                Serial        Data logger
------------------------------------------------------------------------------------------
UM.DDSW..CHZ                Trillium Compact                      00            Gecko
UM.DDSW..CHN                Trillium Compact                      00            Gecko
UM.DDSW..CHE                Trillium Compact                      00            Gecko
AB.WPSH..c03                Guralp CMG-6T seismometer             T6Q24         Echo Pro
XX.LOYU..HH2                -                                     -             -
XX.LOYU..HH1                -                                     -             -
XX.LOYU..004                -                                     -             -
XX.LOYU..001                -                                     -             -
XX.LOYU..HHZ                -                                     -             -
AB.WPNH..c01                Guralp CMG-6T seismometer       

## Print sensitivity for each channel

SUDS provides a single scalar sensitivity (counts / m·s⁻¹).  
Each row below is one channel.

In [6]:
print(f"{'Channel':<24}  {'Sensitivity (counts/(m/s))':.>30}")
print("-" * 56)

for net in inv.networks:
    for sta in net.stations:
        for cha in sta.channels:
            seed_id = f"{net.code}.{sta.code}.{cha.location_code}.{cha.code}"
            if cha.response and cha.response.instrument_sensitivity:
                sens = cha.response.instrument_sensitivity.value
                print(f"{seed_id:<24}  {sens:>30.4f}")
            else:
                print(f"{seed_id:<24}  {'(no response)':>30}")

Channel                   ....Sensitivity (counts/(m/s))
--------------------------------------------------------
UM.DDSW..CHZ                              629145609.3750
UM.DDSW..CHN                              629145609.3750
UM.DDSW..CHE                              629145609.3750
AB.WPSH..c03                             1006632975.0000
XX.LOYU..HH2                                      1.0000
XX.LOYU..HH1                                      1.0000
XX.LOYU..004                                      1.0000
XX.LOYU..001                                      1.0000
XX.LOYU..HHZ                                      1.0000
AB.WPNH..c01                             1006632975.0000
AB.WPNH..c03                             1006632975.0000
AB.WPNH..c02                             1006632975.0000
XX.TRPU..HH1                                      1.0000
XX.TRPU..HH2                                      1.0000
XX.TRPU..HHZ                                      1.0000
XX.TRPU..001                   

## Sanity check — group by sensitivity, show recorder & sensor type\n\nChannels that share a sensitivity value should also share the same recorder and sensor type.\nThis uses the raw STATIONCOMP / INSTRUMENT metadata directly, since ObsPy `Inventory` does not carry those SUDS-specific fields."

In [8]:
from collections import defaultdict
from sudspy.collections import collect_stations, collect_instruments

stations   = collect_stations(PATH)
instruments = collect_instruments(PATH)

# Build per-channel info dict
rows = []
for key, sta in stations.items():
    inst = instruments.get(key)
    cm   = sta["channel_meta"]

    if inst is not None:
        sb      = inst["struct_body"]
        gain_lin = 10 ** (sb.get("gaindb", 0.0) / 20.0)
        sens     = sb["dig_con"] * sb["mot_con"] * gain_lin
        sens_type = sb.get("sens_type", "").strip() or "-"
    else:
        sens      = None
        sens_type = "-"

    rows.append({
        "channel":     key,
        "sensitivity": sens,
        "recorder":    cm.get("recorder", "").strip() or "-",
        "sensor_type": cm.get("sensor_type", "").strip() or "-",   # from STATIONCOMP
        "sens_type":   sens_type,                                   # from INSTRUMENT
    })

# Group by sensitivity value (round to 2 dp to avoid float noise)
groups = defaultdict(list)
for r in rows:
    group_key = round(r["sensitivity"], 2) if r["sensitivity"] is not None else None
    groups[group_key].append(r)

# Print
for sens_val, members in sorted(groups.items(), key=lambda x: (x[0] is None, x[0])):
    print(f"Sensitivity = {sens_val:.4f}" if sens_val is not None else "Sensitivity = (none)")
    print(f"  {'Channel':<24}  recorder  sensor_type(STATIONCOMP)  sens_type(INSTRUMENT)")
    for r in sorted(members, key=lambda x: x["channel"]):
        print(f"  {r['channel']:<24}  {r['recorder']:^8}  {r['sensor_type']:^23}  {r['sens_type']:^20}")
    print()

Sensitivity = 1.0000
  Channel                   recorder  sensor_type(STATIONCOMP)  sens_type(INSTRUMENT)
  XX.LOYU.001                  -                 _                      _          
  XX.LOYU.004                  -                 _                      _          
  XX.LOYU.HH1                  -                 v                      v          
  XX.LOYU.HH2                  -                 v                      v          
  XX.LOYU.HHZ                  -                 v                      v          
  XX.SGWU.001                  -                 _                      _          
  XX.SGWU.HH1                  -                 v                      v          
  XX.SGWU.HH2                  -                 v                      v          
  XX.SGWU.HHZ                  -                 v                      v          
  XX.TRPU.001                  -                 _                      _          
  XX.TRPU.HH1                  -                 v   